In [1]:
!pip install transformers torch

     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     -------------------------------------- 41.5/41.5 kB 976.1 kB/s eta 0:00:00
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/10.1 MB 3.4 MB/s eta 0:00:03
    --------------------------------------- 0.2/10.1 MB 1.8 MB/s eta 0:00:06
   - -------------------------------------- 0.3/10.1 MB 2.1 MB/s eta 0:00:05
   -- ------------------------------------- 0.6/10.1 MB 3.7 MB/s eta 0:00:03
   --- ------------------------------------ 0.9/10.1 MB 4.4 MB/s eta 0:00:03
   ---- ----------------------------------- 1.2/10.1 MB 5.2 MB/s eta 0:00:02
   ----- ---------------------------------- 1.5/10.1 MB 5.3 MB/s eta 0:00:02
   ------- -------------------------------- 2.0/10.1 MB 6.1 MB/s eta 0:00:02
   ------- -------------------------------- 2.0/10.1 MB 6.1 MB/s eta 0:00:02
   ------- -------------------------------- 2.0/10.1 MB 6.1 MB/s eta 0:00:02
   -----


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Import the required classes from the transformers library and PyTorch
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Define the model we want to use. DialoGPT is optimized for multi-turn conversations.
model_name = "microsoft/DialoGPT-medium"

print(f"Loading {model_name}... This might take a minute.")

# Load the tokenizer. The tokenizer converts text into numerical representations (tokens) the model can understand.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the pre-trained transformer model
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

ModuleNotFoundError: No module named 'transformers'

In [ ]:
# Initialize the chat history variable to keep track of the conversation context
chat_history_ids = None

print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
print("(Type 'exit' or 'quit' to end the conversation)\n")

# Use an infinite loop to keep the conversation going until the user exits
while True:
    # 1. Accept user input from the console
    user_input = input("User: ")
    
    # 2. Exit Condition Check
    if user_input.lower() in ['exit', 'quit']:
        print("Chatbot: Goodbye! Have a great day.")
        break
        
    # 3. Encode the user input
    # We add the End-Of-String (eos) token so the model knows a turn has finished, and convert to PyTorch tensors ('pt')
    new_user_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    
    # 4. Maintain Conversation Flow
    # If there is a chat history, concatenate the new input to the old history so the model has context
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_user_input_ids], dim=-1)
    else:
        # If it's the first message, the input is just the new user input
        bot_input_ids = new_user_input_ids

    # 5. Generate Response
    # Pass the concatenated history into the model to generate the next tokens
    chat_history_ids = model.generate(
        bot_input_ids, 
        max_length=1000,              # Maximum length of the entire conversation sequence
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,       # Prevents the model from repeating the same phrases over and over
        do_sample=True,               # Enables probabilistic sampling for more natural, varied responses
        top_k=50,                     # Filters the vocabulary to the top 50 most likely words at each step
        top_p=0.95,                   # Nucleus sampling to ensure high-quality word choices
        temperature=0.7               # Controls randomness (lower = more logical/focused, higher = more creative)
    )

    # 6. Decode and Display Output
    # We only want to decode the *newly generated* tokens, not the entire chat history we fed into the model
    # We slice the tensor from the length of the input tokens to the end
    response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
    
    print(f"Chatbot: {response}")

In [ ]:
hi
